# LAYA crypto trainer on Google Colab

Runs the pipeline on a Colab GPU and keeps **everything that matters on Google Drive**
(`STORAGE`): processed market data, datasets, base and fine-tuned checkpoints, resume states,
predictions, reports and logs. The code and the raw download cache live on the fast, disposable
`/content` disk.

**Every cell can be re-run.** After a disconnect: *Runtime → Run all*. Downloads skip data that is
already in the Drive tables, datasets that exist are reused, finished seeds are not retrained and an
interrupted run continues from its last saved optimizer update (`--auto-resume`).

Before you start: *Runtime → Change runtime type → GPU (T4 or better)*. Optional: add a Hugging Face
read token as a Colab secret named `HF_TOKEN` (key icon on the left) to avoid download rate limits.

Research only: no orders are ever sent. Binance's live API refuses US-hosted Colab machines
(HTTP 451); the historical archive used here is unaffected. Run live `predict` on your own PC.

In [ ]:
#@title Settings
STORAGE = "/content/drive/MyDrive/laya-crypto-storage"  #@param {type:"string"}
BRANCH = "laya-crypto-trainer"  #@param {type:"string"}
TOP_N = 100  #@param {type:"integer"}
DATA_START = "2024-01-01"  #@param {type:"string"}
DATASET = "main"  #@param {type:"string"}
PROFILE = "research"  #@param ["smoke", "research", "deep"]
LEAN_CHECKPOINTS = True  #@param {type:"boolean"}
REPO = "https://github.com/fuadqemberov/crypto-laya.git"
CODE = "/content/laya-crypto-trainer"

In [ ]:
#@title GPU check
!nvidia-smi || echo "No GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
#@title Mount Drive, set storage locations
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(STORAGE, exist_ok=True)
os.environ["LAYA_CRYPTO_STORAGE"] = STORAGE                 # data, models, reports, artifacts, logs
os.environ["LAYA_CRYPTO_RAW_DIR"] = "/content/raw_cache"    # disposable zip cache (local disk)
os.environ["LAYA_CRYPTO_CONFIG_DIR"] = f"{STORAGE}/configs" # your config edits persist on Drive
os.environ["LAYA_CRYPTO_LEAN"] = "1" if LEAN_CHECKPOINTS else "0"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
u = shutil.disk_usage(STORAGE)
print(f"Drive free: {u.free / 2**30:.1f} GB (a research run needs ~7 GB lean, ~12 GB otherwise)")

In [ ]:
#@title Get the code and install (Colab's CUDA PyTorch is kept)
import os
if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} checkout -q {BRANCH} && git -C {CODE} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO} {CODE}
!pip install -q -e {CODE}
!git -C {CODE} log -1 --format="code at %h %cd"

In [ ]:
#@title Configs on Drive (created once; edit them there, they survive sessions)
import os, shutil, yaml
cfg_dir = os.environ["LAYA_CRYPTO_CONFIG_DIR"]
if not os.path.isdir(cfg_dir):
    shutil.copytree(f"{CODE}/configs", cfg_dir)
path = f"{cfg_dir}/data.yaml"
with open(path) as f:
    data_cfg = yaml.safe_load(f)
data_cfg["start"] = DATA_START
data_cfg.setdefault("universe", {})["top_n"] = TOP_N
with open(path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open(path).read()[:600])

In [ ]:
#@title Helper
from IPython import get_ipython
def cli(cmd):
    ip = get_ipython()
    ip.system(f"cd {CODE} && python -m laya_crypto.cli {cmd}")
    if ip.user_ns.get("_exit_code"):
        raise RuntimeError(f"command failed: {cmd}")
def exists(*parts):
    return os.path.exists(os.path.join(STORAGE, *parts))

In [ ]:
#@title 1. Hardware report and the official LAYA base checkpoint (downloaded once to Drive)
cli("hardware")
cli("laya-demo")

In [ ]:
#@title 2. Point-in-time universe and market data (incremental: re-running only fetches what is new)
if not exists("data", "universe", "universe.parquet"):
    cli(f"universe --top-n {TOP_N}")
cli("download")
cli("validate --timeframe 5m")

In [ ]:
#@title 3. Dataset (immutable: built once per name)
if not exists("data", "datasets", DATASET, "manifest.json"):
    cli(f"dataset --name {DATASET}")
else:
    print(f"dataset {DATASET!r} already on Drive; use a new DATASET name to rebuild")

In [ ]:
#@title 4. Baselines and walk-forward (fast; LAYA must beat these)
if not exists("reports", DATASET, "lightgbm.json"):
    cli(f"baselines --dataset {DATASET}")
if not exists("reports", DATASET, "walk_forward_lightgbm.json"):
    cli(f"walk-forward --dataset {DATASET} --model lightgbm")

In [ ]:
#@title 5. Fine-tune LAYA (re-run this cell after a disconnect: it continues where it stopped)
cli(f"train --profile {PROFILE} --dataset {DATASET} --device cuda --auto-resume")

In [ ]:
#@title 6. Select the best seed (validation only), compare, backtest
cli(f"select-best --dataset {DATASET} --profile {PROFILE}")
import json, glob
best = json.load(open(f"{STORAGE}/models/best/selection.json"))["run_id"]
cli(f"backtest --dataset {DATASET} --model lightgbm --split test")
cli(f"backtest --dataset {DATASET} --model laya-{best} --split test")
cli(f"evaluate --dataset {DATASET}")

In [ ]:
#@title 7. Show the comparison report
from IPython.display import Markdown, display
display(Markdown(open(f"{STORAGE}/reports/{DATASET}/comparison.md").read()))

In [ ]:
#@title 8. Make sure Drive has everything written before you close the tab
drive.flush_and_unmount()
print("Drive flushed. Re-run the 'Mount Drive' cell to continue working.")

## Using the Colab results on your PC

Install *Google Drive for desktop*, let it sync `laya-crypto-storage`, then in PowerShell from the
project folder:

```powershell
$env:LAYA_CRYPTO_STORAGE = "G:\My Drive\laya-crypto-storage"   # your Drive letter/path
uv run python -m laya_crypto.cli dashboard
uv run python -m laya_crypto.cli predict --symbol BTCUSDT
```

The dashboard then shows the Colab runs, reports and backtests, and `predict` uses `models\best`
from Drive with live Binance data from your own connection.